In [0]:
%pip install -U \
    mlflow[databricks] \
    -U langgraph \
    langchain \
    langchain-core \
    databricks-langchain \
    databricks-ai-search \
    databricks-sdk \
    databricks-agents \
    -U ddgs

dbutils.library.restartPython()

In [0]:
import mlflow

# Set experiment for better organization
mlflow.set_experiment("/Users/naval.datamaster@gmail.com/resume-rag-agent")

# Enable autologging BEFORE creating the LangChain client
mlflow.langchain.autolog()

In [0]:
CATALOG = "dev"
SCHEMA = "bronze"

RESUME_TABLE = f"{CATALOG}.{SCHEMA}.resume_chunks"

AI_SEARCH_ENDPOINT = "vector_db"

AI_SEARCH_INDEX = (
    f"{CATALOG}.{SCHEMA}.resume_index"
)

LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

AGENT_MODEL_NAME = (
    f"{CATALOG}.{SCHEMA}.resume_rag_agent"
)

#MLFLOW_EXPERIMENT = "/Shared/resume-rag-agent"

In [0]:
display(
    spark.table(RESUME_TABLE)
)

In [0]:
from databricks.ai_search.client import AISearchClient

client = AISearchClient()

index = client.get_index(
    index_name=AI_SEARCH_INDEX
)

index.describe()

In [0]:
results = index.similarity_search(
    query_text="Python Spark Databricks experience",
    columns=[
        "candidate_id",
        "source_file",
        "page_number",
        "content"
    ],
    num_results=5,
    query_type="hybrid"
)

results

In [0]:
from langchain_core.tools import tool
from databricks.ai_search.client import AISearchClient

ai_search_client = AISearchClient()

resume_index = ai_search_client.get_index(
    index_name=AI_SEARCH_INDEX
)

In [0]:
@tool
def search_resumes(query: str, num_results: int = 5) -> str:
    """
    Search the resume knowledge base.

    Use this tool whenever the user asks about candidate
    experience, skills, education, projects, certifications,
    technologies, or resume content.
    """

    results = resume_index.similarity_search(
        query_text=query,
        columns=[
            "candidate_id",
            "source_file",
            "page_number",
            "content"
        ],
        num_results=num_results,
        query_type="hybrid"
    )

    rows = results["result"]["data_array"]

    if not rows:
        return "No relevant resume information was found."

    formatted = []

    for row in rows:
        candidate_id = row[0]
        source_file = row[1]
        page_number = row[2]
        content = row[3]

        formatted.append(
            f"""
Candidate: {candidate_id}
Source: {source_file}
Page: {page_number}

Content:
{content}
"""
        )

    return "\n---\n".join(formatted)

In [0]:
from langchain_core.tools import tool

@tool
def get_candidate_resume(candidate_id: str) -> str:
    """
    Retrieve all available resume information for a specific candidate.
    """

    rows = (
        spark.table(RESUME_TABLE)
        .filter(f"candidate_id = '{candidate_id}'")
        .orderBy("page_number")
        .select(
            "candidate_id",
            "source_file",
            "page_number",
            "content"
        )
        .collect()
    )

    if not rows:
        return f"No resume found for candidate {candidate_id}."

    output = []

    for row in rows:
        output.append(
            f"""
Candidate: {row.candidate_id}
File: {row.source_file}
Page: {row.page_number}

{row.content}
"""
        )

    return "\n---\n".join(output)

In [0]:
from ddgs import DDGS
from langchain_core.tools import tool


@tool
def web_search(query: str, max_results: int = 5) -> str:
    """
    Search the public web using DuckDuckGo.

    Use this tool when the user asks for current external information,
    company information, technology information, job requirements,
    industry information, or anything that may not exist in the
    internal resume knowledge base.
    """

    results = DDGS().text(
        query,
        max_results=max_results
    )

    if not results:
        return "No web search results found."

    formatted_results = []

    for result in results:
        formatted_results.append(
            f"""
Title: {result.get('title', '')}

URL: {result.get('href', '')}

Snippet:
{result.get('body', '')}
"""
        )

    return "\n---\n".join(formatted_results)

In [0]:
result = web_search.invoke(
    {
        "query": "Databricks Data Engineer skills 2026"
    }
)

print(result)

In [0]:
from databricks_langchain import ChatDatabricks
from langchain.agents import create_agent

llm = ChatDatabricks(
    endpoint=LLM_ENDPOINT,
    temperature=0
)

In [0]:
tools = [
    search_resumes,
    get_candidate_resume,
    web_search
]

In [0]:
graph = create_agent(
    model=llm,
    tools=tools
)

In [0]:
response = graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Find candidates with strong Databricks and PySpark experience."
            }
        ]
    }
)

response